# 풍력발전량 예측 - 추론

`model_artifacts/`의 모델로 2025년 8,760시간을 예측하고, 후처리를 거쳐 `open/final_submission.csv`를 만든다.
모델 파일이 없으면 `submit_train.ipynb`를 먼저 실행한다. 10~15초 정도 걸린다.

## 0. 설정

노트북은 `open/` 폴더가 있는 위치에서 실행한다.

In [1]:
import hashlib
import io
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# 노트북이 있는 폴더(open/ 의 상위 폴더)를 기준 경로로 쓴다.
# 다른 위치에서 실행한다면 여기만 바꾸면 된다.
NB_ROOT = Path.cwd()
assert (NB_ROOT / "open").is_dir(), (
    f"open/ 을 찾을 수 없습니다: {NB_ROOT}\n"
    "노트북을 프로젝트 루트에서 실행하거나 NB_ROOT 를 직접 지정하세요.")

# cp949 콘솔에서 스크립트로 돌릴 때 출력 인코딩 오류 방지 (노트북에서는 건너뜀)
if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8")

## 1. 피처 생성

학습 노트북과 같은 코드.

In [2]:
DATA_DIR = NB_ROOT / "open"

# 외부 예보 파일 선택. EXT_SUFFIX 가 있으면 external_{name}{EXT_SUFFIX}.csv 를 읽고,
# None 이면 COMPLIANT_EXT 값에 따라 _compliant 파일 또는 원본 파일을 읽는다.
COMPLIANT_EXT = True
EXT_SUFFIX = "_a1"

USE_MSM = True  # JMA MSM 지상풍 피처 (external_msm.csv)

def ext_csv(name):
    suf = EXT_SUFFIX if EXT_SUFFIX is not None else ("_compliant" if COMPLIANT_EXT else "")
    return DATA_DIR.parent / f"external_{name}{suf}.csv"

def day_block(t):
    # 발행 블록: D 01:00 ~ D+1 00:00 은 D-1 에 나온 같은 예보다.
    # lead, 중심 롤링처럼 뒤 시각을 보는 피처는 이 블록 안에서만 계산한다.
    # lag, diff 는 더 이전에 나온 값이라 제한하지 않는다.
    return (t - pd.Timedelta(hours=1)).dt.normalize()

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]

CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}

FARM_LAT, FARM_LON = 37.280, 128.963  # 터빈 위치 평균 (info.xlsx)

# (u 컬럼, v 컬럼, 풍속 컬럼 이름)
UV_PAIRS = {
    "ldaps": [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "ws10"),
        ("heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax", "ws50max"),
        ("heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin", "ws50min"),
        ("heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS", "ws5bl"),
    ],
    "gfs": [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "ws10"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "ws80"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "ws100"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "wspbl"),
        ("isobaricInhPa_850_u", "isobaricInhPa_850_v", "ws850"),
    ],
}

# 세제곱 피처를 추가할 풍속 (허브 높이 근처)
CUBED = {"ldaps": {"ws50max", "ws50min"}, "gfs": {"ws80", "ws100"}}

META_COLS = {"forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"}

# LDAPS 50MUmax/MVmax 는 성분별 최대값이라 벡터 크기 차이로 만든 spread 가 음수가 되기도 한다.
# True 면 u/v 변동폭으로 다시 계산한다 (피처 이름과 개수는 그대로). 최종 모델은 False.
SPREAD_FIX = False

def add_wind_speed(df, prefix):
    # 풍속은 격자별로 계산한 뒤 집계한다 (u/v 를 먼저 평균하면 방향이 상쇄돼 작아짐)
    for u_col, v_col, name in UV_PAIRS[prefix]:
        ws = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        df[name] = ws
        if name in CUBED[prefix]:
            df[f"{name}_cube"] = ws ** 3
    if SPREAD_FIX and prefix == "ldaps":
        du = df["heightAboveGround_50_50MUmax"] - df["heightAboveGround_50_50MUmin"]
        dv = df["heightAboveGround_50_50MVmax"] - df["heightAboveGround_50_50MVmin"]
        df["ws50osc"] = np.sqrt(du ** 2 + dv ** 2) / 2
    return df

# ldaps_test.csv 의 세 시각(2025-04-08 17시, 06-18 18시, 07-18 06시)은 16개 격자가 전부 비어 있다.
# 학습 데이터에는 결측이 없어서 모델이 이 NaN 을 0 처럼 다루게 되므로, 같은 발행 블록 안에서
# 시간 보간으로 채운다. 학습 데이터에는 영향이 없다.
IMPUTE_BLOCK_GAPS = True


def fill_block_gaps(df):
    """발행 블록 안에서 격자별로 시간 선형보간. 결측이 없으면 그대로 반환."""
    value_cols = [c for c in df.columns if c not in META_COLS]
    if not df[value_cols].isna().to_numpy().any():
        return df
    df = df.sort_values(["grid_id", "forecast_kst_dtm"])
    df[value_cols] = df.groupby(["grid_id", "data_available_kst_dtm"])[value_cols].transform(
        lambda s: s.interpolate(limit_direction="both"))
    return df.sort_index()


def prep(csv_path, prefix):
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    if IMPUTE_BLOCK_GAPS:
        df = fill_block_gaps(df)
    return add_wind_speed(df, prefix)

def agg_mean(df, prefix, suffix="mean"):
    value_cols = [c for c in df.columns if c not in META_COLS]
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}_{suffix}" for c in agg.columns]
    return agg.reset_index()

def agg_std(df, prefix, cols):
    agg = df.groupby("forecast_kst_dtm")[cols].std()
    agg.columns = [f"{prefix}_{c}_std" for c in agg.columns]
    return agg.reset_index()

def nearest_grids(df, k):
    g = df[["grid_id", "latitude", "longitude"]].drop_duplicates("grid_id")
    d2 = (g["latitude"] - FARM_LAT) ** 2 + (g["longitude"] - FARM_LON) ** 2
    return g.loc[d2.nsmallest(k).index, "grid_id"].tolist()

def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out

def build_weather(ldaps_csv, gfs_csv):
    ldaps = prep(ldaps_csv, "ldaps")
    gfs = prep(gfs_csv, "gfs")

    # 전체 격자 평균 + 단지 인근 격자 평균(LDAPS 4개, GFS 1개) + 격자 간 표준편차
    base = agg_mean(ldaps, "ldaps").merge(agg_mean(gfs, "gfs"), on="forecast_kst_dtm", how="inner")
    near = agg_mean(ldaps[ldaps["grid_id"].isin(nearest_grids(ldaps, 4))], "ldaps", "n4").merge(
        agg_mean(gfs[gfs["grid_id"].isin(nearest_grids(gfs, 1))], "gfs", "n1"),
        on="forecast_kst_dtm", how="inner",
    )
    std = agg_std(ldaps, "ldaps", ["ws10", "ws50max", "ws5bl"]).merge(
        agg_std(gfs, "gfs", ["ws10", "ws80", "ws100"]), on="forecast_kst_dtm", how="inner"
    )
    w = base.merge(near, on="forecast_kst_dtm", how="left").merge(std, on="forecast_kst_dtm", how="left")

    # 시계열 피처 (lag/lead/diff/rolling). 뒤 시각을 보는 것은 발행 블록 안에서만.
    w = w.sort_values("forecast_kst_dtm").reset_index(drop=True)
    blk = day_block(w["forecast_kst_dtm"])
    for c in ["ldaps_ws50max_n4", "gfs_ws100_n1", "gfs_ws80_n1", "ldaps_ws50max_mean", "gfs_ws100_mean"]:
        s = w[c]
        for lag in (1, 2, 3):
            w[f"{c}_lag{lag}"] = s.shift(lag)
            w[f"{c}_lead{lag}"] = s.groupby(blk).shift(-lag)
        w[f"{c}_diff1"] = s - s.shift(1)
        w[f"{c}_roll6_mean"] = s.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).mean())
        w[f"{c}_roll6_std"] = s.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).std())
    for c in ["ldaps_ws50max_n4", "gfs_ws100_n1", "gfs_wspbl_n1", "ldaps_ws10_n4"]:
        s = w[c]
        for win in (12, 24):
            w[f"{c}_roll{win}_mean"] = s.groupby(blk).transform(
                lambda x: x.rolling(win, center=True, min_periods=1).mean())
            w[f"{c}_roll{win}_std"] = s.groupby(blk).transform(
                lambda x: x.rolling(win, center=True, min_periods=1).std())
        w[f"{c}_diff3"] = s - s.shift(3)
        w[f"{c}_lead6"] = s.groupby(blk).shift(-6)
        w[f"{c}_lag6"] = s.shift(6)

    # 50m 풍속 spread, 100m 풍향, 대기 안정도(850hPa - 2m 기온), 하루 풍속 요약
    if SPREAD_FIX:
        w["ws50_spread_n4"] = w["ldaps_ws50osc_n4"]
        w["ws50_spread_mean"] = w["ldaps_ws50osc_mean"]
        w = w.drop(columns=["ldaps_ws50osc_n4", "ldaps_ws50osc_mean"])
    else:
        w["ws50_spread_n4"] = w["ldaps_ws50max_n4"] - w["ldaps_ws50min_n4"]
        w["ws50_spread_mean"] = w["ldaps_ws50max_mean"] - w["ldaps_ws50min_mean"]
    ws100 = np.sqrt(w["gfs_heightAboveGround_100_100u_n1"] ** 2 + w["gfs_heightAboveGround_100_100v_n1"] ** 2)
    w["wd100_sin_n1"] = w["gfs_heightAboveGround_100_100u_n1"] / ws100.replace(0, np.nan)
    w["wd100_cos_n1"] = w["gfs_heightAboveGround_100_100v_n1"] / ws100.replace(0, np.nan)
    w["t_grad_n1"] = w["gfs_isobaricInhPa_850_t_n1"] - w["gfs_heightAboveGround_2_2t_n1"]
    day = (w["forecast_kst_dtm"] - pd.Timedelta(hours=1)).dt.date
    g = w.groupby(day)["gfs_ws100_n1"]
    w["ws100_day_mean"] = g.transform("mean")
    w["ws100_day_max"] = g.transform("max")
    w["ws100_day_min"] = g.transform("min")
    g2 = w.groupby(day)["ldaps_ws50max_n4"]
    w["ws50_day_mean"] = g2.transform("mean")
    w["ws50_day_max"] = g2.transform("max")
    return w

def add_icon_features(df):
    # DWD ICON 예보. 2022-11-24 이전은 NaN 이고 LightGBM 이 그대로 처리한다.
    ic = pd.read_csv(ext_csv("icon"), encoding="utf-8-sig", parse_dates=["time"])
    ic = ic.set_index("time")
    t = df["forecast_kst_dtm"]
    out = pd.DataFrame(index=df.index)
    ws100 = ic["wind_speed_100m"].reindex(t).to_numpy()
    wd = np.deg2rad(ic["wind_direction_100m"].reindex(t).to_numpy())
    out["icon_ws100"] = ws100
    out["icon_ws100_cube"] = ws100 ** 3
    out["icon_ws10"] = ic["wind_speed_10m"].reindex(t).to_numpy()
    out["icon_gust"] = ic["wind_gusts_10m"].reindex(t).to_numpy()
    out["icon_wd_sin"] = np.sin(wd)
    out["icon_wd_cos"] = np.cos(wd)
    out["icon_t2m"] = ic["temperature_2m"].reindex(t).to_numpy()
    out["icon_sp"] = ic["surface_pressure"].reindex(t).to_numpy()
    out["icon_minus_gfs_ws100"] = ws100 - df["gfs_ws100_n1"].to_numpy()
    out["icon_minus_ldaps_ws50"] = ws100 - df["ldaps_ws50max_n4"].to_numpy()
    s = pd.Series(ws100, index=df.index)
    blk = day_block(df["forecast_kst_dtm"])
    out["icon_ws100_roll6_mean"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).mean())
    out["icon_ws100_roll6_std"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).std())
    out["icon_ws100_diff1"] = s - s.shift(1)
    return pd.concat([df, out], axis=1)

def add_ext_features(df):
    # GEM(120m/80m), UKMO/JMA(10m) 예보와 모델 간 차이.
    # UKMO, JMA 는 2022년 초부터 있어서 ICON/GEM 이 없는 2022년 구간을 채워 준다.
    ic = pd.read_csv(ext_csv("icon"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    ge = pd.read_csv(ext_csv("gem"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    uk = pd.read_csv(ext_csv("ukmo"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    jm = pd.read_csv(ext_csv("jma"), encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
    t = df["forecast_kst_dtm"]
    out = pd.DataFrame(index=df.index)

    ws120 = ge["wind_speed_120m"].reindex(t).to_numpy()
    gwd = np.deg2rad(ge["wind_direction_120m"].reindex(t).to_numpy())
    out["gem_ws120"] = ws120
    out["gem_ws120_cube"] = ws120 ** 3
    out["gem_ws80"] = ge["wind_speed_80m"].reindex(t).to_numpy()
    out["gem_gust"] = ge["wind_gusts_10m"].reindex(t).to_numpy()
    out["gem_wd_sin"] = np.sin(gwd)
    out["gem_wd_cos"] = np.cos(gwd)
    out["gem_minus_icon"] = ws120 - ic["wind_speed_100m"].reindex(t).to_numpy()
    out["gem_minus_gfs"] = ws120 - df["gfs_ws100_n1"].to_numpy()
    s = pd.Series(ws120, index=df.index)
    blk = day_block(df["forecast_kst_dtm"])
    out["gem_ws120_roll6_mean"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).mean())
    out["gem_ws120_roll6_std"] = s.groupby(blk).transform(
        lambda x: x.rolling(6, center=True, min_periods=1).std())
    out["gem_ws120_diff1"] = s - s.shift(1)

    icon10 = ic["wind_speed_10m"].reindex(t).to_numpy()
    uws = uk["wind_speed_10m"].reindex(t).to_numpy()
    jws = jm["wind_speed_10m"].reindex(t).to_numpy()
    uwd = np.deg2rad(uk["wind_direction_10m"].reindex(t).to_numpy())
    jwd = np.deg2rad(jm["wind_direction_10m"].reindex(t).to_numpy())
    out["ukmo_ws10"] = uws
    out["ukmo_gust"] = uk["wind_gusts_10m"].reindex(t).to_numpy()
    out["jma_ws10"] = jws
    out["ukmo_wd_sin"] = np.sin(uwd)
    out["ukmo_wd_cos"] = np.cos(uwd)
    out["jma_wd_sin"] = np.sin(jwd)
    out["jma_wd_cos"] = np.cos(jwd)
    out["ukmo_minus_jma"] = uws - jws
    out["ukmo_minus_icon10"] = uws - icon10
    out["jma_minus_icon10"] = jws - icon10

    if USE_MSM:
        # JMA MSM 은 전날 00UTC 실행분만 모아 둔 자료라 _a1 전처리 없이 바로 읽는다
        ms = pd.read_csv(DATA_DIR.parent / "external_msm.csv", encoding="utf-8-sig", parse_dates=["time"]).set_index("time")
        ms = ms[~ms.index.duplicated(keep="last")].sort_index()
        mws = ms["msm_ws10"].reindex(t).to_numpy()
        mwd = np.deg2rad(ms["msm_wd10"].reindex(t).to_numpy())
        out["msm_ws10"] = mws
        out["msm_wd_sin"] = np.sin(mwd)
        out["msm_wd_cos"] = np.cos(mwd)
        out["msm_minus_jma"] = mws - jws
        out["msm_minus_icon10"] = mws - icon10
        sm = pd.Series(mws, index=df.index)
        out["msm_ws10_roll6_mean"] = sm.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).mean())
        out["msm_ws10_roll6_std"] = sm.groupby(blk).transform(
            lambda x: x.rolling(6, center=True, min_periods=1).std())
    return pd.concat([df, out], axis=1)

## 2. 예측과 후처리

그룹별 모델과 통합 모델 예측을 0.3 : 0.7로 섞고 isotonic 보정, 구간 오프셋을 적용한 뒤 `post_transform`으로 후처리한다.
후처리 단계별 설명은 README 참고.

In [3]:
def apply_bin_shifts(p, edges, deltas, cap):
    bin_id = np.clip(np.digitize(p, edges[1:-1]), 0, len(deltas) - 1)
    return np.clip(p + deltas[bin_id], 0, cap)


ART = NB_ROOT / "model_artifacts"
OUT = DATA_DIR / "final_submission.csv"
CHAMPION = DATA_DIR / "probe_month_g1m10s95.csv"  # 최종 제출 파일. 있으면 결과와 비교한다

# ---- 후처리 상수 ----
# 대부분 학습 기간 홀드아웃으로 방향을 잡고, 값은 Public LB 점수를 보면서 정했다.

# 그룹별 레벨 배율. 평가 기간 예측 평균으로 계산하면 미래 정보가 섞이므로 고정값으로 둔다.
LEVEL_K = {"kpx_group_1": 0.979264933018213,
           "kpx_group_2": 1.003054037825868,
           "kpx_group_3": 0.853742381189483}
G3_WIN_BASE, G3_JAN_EXTRA, G3_FEB_EXTRA = 0.95, 0.79, 0.76   # g3 겨울 구간 배율
T3_ADD, T3_LO, T3_HI = 0.045, 0.04, 0.14                     # 저출력 구간 가산
G1_TAU, G1_PLATEAU = 0.89, 20100.0                           # 상단 스냅: τ 이상이면 최소 이 값
G2_TAU, G2_PLATEAU = 0.85, 19569.0
G3_TAU, G3_FLOOR = 0.78, 17640.0                             # 84%cap, 실제로는 12월에만 적용됨
G3_CEIL_PCT, G3_CEIL_M1, G3_CEIL_M2 = 0.80, 1, 11            # g3 상한 (1~11월)
G3_PLAT_TAU2 = 0.7760                                        # 이 값 이상이면 g3 상한값으로 맞춤
# 발행 블록 시각별 배율. early = 블록 시각 13시 이하 (00시는 24시로 본다).
# 13시는 previous_day1 을 쓸 수 있는 경계라서 정해진 값이고, 튜닝하지 않았다.
G12_EARLY_MULT = 1.02                                        # g1, g2 early
G3_EARLY_MULT, G3_LATE_MULT = 1.03, 1.02
PLATEAU_KWH = {"kpx_group_1": G1_PLATEAU, "kpx_group_2": G2_PLATEAU}
ELIG_FLOOR = 0.10                                            # 예측 하한 (설비용량 비율)

# 그룹별 달력 구간 배율: (그룹, 월, 시작일, 종료일, 배율)
# 후처리 맨 마지막에 적용한다. 같은 그룹, 같은 달의 구간은 서로 겹치지 않아야 한다.
CAL_WINDOWS = [
    ("kpx_group_2", 1,  1, 15, 0.85),
    ("kpx_group_2", 1, 16, 31, 0.95),
    ("kpx_group_2", 7,  1, 31, 0.95),
    ("kpx_group_2", 8,  1, 31, 0.95),
    ("kpx_group_2", 9,  1, 30, 0.95),
    ("kpx_group_1", 3,  1, 31, 0.90),
    ("kpx_group_1", 4,  1, 30, 0.90),
    ("kpx_group_1", 5,  1, 31, 0.90),
    ("kpx_group_1", 6,  1, 30, 0.90),
    ("kpx_group_1", 10, 1, 31, 0.95),
]


def build_test_matrix(X_columns):
    """평가 기간 피처 행렬. 학습 때와 같은 함수, 같은 열 순서를 쓴다."""
    ss = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    ss["forecast_kst_dtm"] = pd.to_datetime(ss["forecast_kst_dtm"])
    weather = build_weather(DATA_DIR / "test" / "ldaps_test.csv",
                               DATA_DIR / "test" / "gfs_test.csv")
    test_df = ss[["forecast_id", "forecast_kst_dtm"]].merge(
        weather, on="forecast_kst_dtm", how="left")
    test_df = add_ext_features(add_icon_features(test_df))
    X_test = pd.concat(
        [calendar_features(test_df["forecast_kst_dtm"]),
         test_df.drop(columns=["forecast_id", "forecast_kst_dtm"])],
        axis=1,
    )[X_columns]
    return ss, X_test


def predict_raw(X_test, schema):
    """그룹별/통합 모델 예측 -> 블렌드 -> isotonic 보정 -> 구간 오프셋."""
    with open(ART / "models_pool.pkl", "rb") as f:
        pool_models = pickle.load(f)
    with open(ART / "calibrators.pkl", "rb") as f:
        calib = pickle.load(f)

    # 통합 모델은 group_id 만 바꿔 가며 3개 그룹을 예측한다
    pool_test = {}
    Xt = X_test.copy()
    for gi, g in enumerate(schema["target_cols"]):
        Xt["group_id"] = gi
        pool_test[g] = np.mean([m.predict(Xt[schema["pool_columns"]]) for m in pool_models],
                               axis=0) * schema["capacity"][g]

    out = {}
    alpha = schema["pooled_alpha"]
    for g in schema["target_cols"]:
        cap = schema["capacity"][g]
        with open(ART / f"models_grp_{g}.pkl", "rb") as f:
            models = pickle.load(f)
        grp = np.mean([m.predict(X_test) for m in models], axis=0)
        pred = np.clip((1 - alpha) * grp + alpha * pool_test[g], 0, cap)
        c = calib[g]
        pred = c["iso"].predict(pred)
        pred = apply_bin_shifts(pred, c["edges"], c["deltas"], cap)
        out[g] = pred
        print(f"  {g}: 평균 {pred.mean() / cap:6.2%}cap  최대 {pred.max() / cap:6.2%}cap")
    return out


def post_transform(ss, raw):
    """고정 상수만 쓰는 후처리. 학습이나 피팅은 없다.

    순서
      1. 레벨 배율          2. g3 겨울 구간      3. 롤링 중앙값
      4. 저출력 구간 가산   5. 상단 스냅         6. g3 상한
      7. g3 고원            8. g1/g2 early 배율  9~10. g3 early/late 배율
      11. 하한              12. 달력 구간 배율

    순서를 바꾸면 결과가 달라진다. 특히 11번 하한은 배율 단계들보다 뒤에 있어야 하고,
    12번에서 배율을 곱한 뒤에는 하한을 다시 적용한다.
    """
    t = pd.to_datetime(ss["forecast_kst_dtm"])
    tt = t - pd.Timedelta(hours=1)          # kst_dtm 은 집계 구간의 끝 시각
    month, day = tt.dt.month, tt.dt.day
    hr = t.dt.hour.to_numpy()
    early = np.where(hr == 0, 24, hr) <= 13   # 발행 블록 기준 시각 (00시는 24시)
    jan_win = (month == 1) & (day >= 19)
    feb_win = (month == 2) & (day <= 10)
    blk = tt.dt.normalize()                 # 발행 블록

    sub = ss[["forecast_id", "forecast_kst_dtm"]].copy()
    for g, cap in CAPACITY_KWH.items():
        cap = float(cap)
        p = pd.Series(raw[g])

        # 1. 레벨 배율
        p = (p * LEVEL_K[g]).clip(0, cap)

        # 2. g3 겨울 구간 (1/19~1/31, 2/1~2/10)
        if g == "kpx_group_3":
            p = p.where(~(jan_win | feb_win), (p * G3_WIN_BASE).clip(0, cap))
            p = p.where(~jan_win, (p * G3_JAN_EXTRA).clip(0, cap))
            p = p.where(~feb_win, (p * G3_FEB_EXTRA).clip(0, cap))

        # 3. 발행 블록 안에서 3시간 중심 롤링 중앙값.
        #    블록 마지막 시각은 다음 시각을 볼 수 없어서 직전 2개 값의 중앙값으로 대신한다.
        med = p.groupby(blk).transform(lambda x: x.rolling(3, center=True, min_periods=1).median())
        med2 = p.rolling(2, min_periods=1).median()
        is_last = blk != blk.shift(-1)
        v = med.where(~is_last, med2).to_numpy().copy()

        # 4. 저출력 구간(4~14%cap) 가산
        sel = (v >= T3_LO * cap) & (v < T3_HI * cap)
        v[sel] += T3_ADD * cap
        v_pre = v.copy()        # 7번 고원 판정에 쓰는 값 (스냅, 상한 적용 전)

        # 5. 상단 스냅: 정격 근처 예측이 눌려 있는 것을 풀어 준다
        if g == "kpx_group_1":
            hi = v >= G1_TAU * cap
            v[hi] = np.maximum(v[hi], G1_PLATEAU)
        elif g == "kpx_group_2":
            hi = v >= G2_TAU * cap
            v[hi] = np.maximum(v[hi], G2_PLATEAU)
        elif g == "kpx_group_3":
            # g3 는 1~11월이 6번 상한(80%cap)에 덮이므로 사실상 12월에만 작동한다
            hi = v >= G3_TAU * cap
            v[hi] = np.maximum(v[hi], G3_FLOOR)

        v = np.clip(v, 0, cap)

        # 6. g3 상한 80%cap (1~11월).
        #    학습 데이터 끝(2024-12)에 unison_wtg01 이 계속 멈춰 있어서 5기 중 4기 용량으로 잡았다.
        if g == "kpx_group_3":
            m = ((month >= G3_CEIL_M1) & (month <= G3_CEIL_M2)).to_numpy()
            v[m] = np.minimum(v[m], G3_CEIL_PCT * cap)

            # 7. 고원: 스냅 전 값이 τ2 이상이면 상한값에 붙인다
            v[m & (v_pre >= G3_PLAT_TAU2 * cap)] = G3_CEIL_PCT * cap

        # 8. g1/g2 early 배율. 이미 고원값 이상인 시간은 그대로 둔다.
        if g in PLATEAU_KWH:
            tgt = early & (v < PLATEAU_KWH[g])
            v[tgt] = np.clip(v[tgt] * G12_EARLY_MULT, 0.0, cap)

        # 9~10. g3 early/late 배율. 1~11월은 배율 후 다시 상한으로 자른다.
        if g == "kpx_group_3":
            scaled = v * G3_EARLY_MULT
            v9 = np.where(m, np.minimum(scaled, G3_CEIL_PCT * cap), scaled)
            v = np.where(early, np.clip(v9, 0.0, cap), v)

            # late 는 12월 스냅 값(G3_FLOOR)에 걸린 시간은 건드리지 않는다
            frz = ~m & (np.abs(v - G3_FLOOR) < 1e-9)
            scaled = np.where(frz, v, v * G3_LATE_MULT)
            v10 = np.where(m, np.minimum(scaled, G3_CEIL_PCT * cap), scaled)
            v = np.where(~early, np.clip(v10, 0.0, cap), v)

        # 11. 하한 10%cap. 실측이 10%cap 이상인 시간만 채점하므로
        #     예측을 그 아래로 두면 손해만 있고 이득이 없다.
        v = np.maximum(v, ELIG_FLOOR * cap)

        # 12. 달력 구간 배율. 곱한 뒤 하한 아래로 내려간 값은 다시 올린다.
        for wg, wm, wd0, wd1, ws in CAL_WINDOWS:
            if wg != g:
                continue
            wsel = ((month == wm) & (day >= wd0) & (day <= wd1)).to_numpy()
            v = np.where(wsel, np.clip(v * ws, 0.0, cap), v)
        if any(wg == g for wg, *_ in CAL_WINDOWS):
            v = np.maximum(v, ELIG_FLOOR * cap)

        sub[g] = v
    return sub


def verify(sub, ss):
    """제출 파일 형식 확인."""
    assert len(sub) == 8760, f"행 수 {len(sub)}"
    assert (sub["forecast_id"].values == ss["forecast_id"].values).all()
    assert sub[list(CAPACITY_KWH)].notna().all().all()
    assert np.isfinite(sub[list(CAPACITY_KWH)].to_numpy()).all()
    for g, cap in CAPACITY_KWH.items():
        assert sub[g].min() >= 0 and sub[g].max() <= cap, f"{g} 범위 위반"
    print("  형식 확인: 8,760행, id 일치, 결측 없음, 0 ~ 설비용량 범위")


def main():
    t0 = time.time()
    if not (ART / "schema.pkl").exists():
        raise SystemExit(f"{ART} 가 없습니다. submit_train.ipynb 를 먼저 실행하세요.")
    with open(ART / "schema.pkl", "rb") as f:
        schema = pickle.load(f)

    print("=" * 70)
    print("추론 코드 (inference) — 학습 없음, 저장된 모델만 사용")
    print("=" * 70)

    print("\n[1/4] 평가 기간 피처 구축...")
    ss, X_test = build_test_matrix(schema["X_columns"])
    print(f"  X_test: {X_test.shape}")

    print("\n[2/4] 모델 예측 + 보정층...")
    raw = predict_raw(X_test, schema)

    print("\n[3/4] 사후 변환 체인...")
    sub = post_transform(ss, raw)
    for g, cap in CAPACITY_KWH.items():
        print(f"  {g}: 평균 {sub[g].mean() / cap:6.2%}cap  최대 {sub[g].max() / cap:6.2%}cap")

    print("\n[4/4] 검증 및 저장...")
    sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    verify(sub, ss)
    sub.to_csv(OUT, index=False, encoding="utf-8-sig")
    h = hashlib.sha256(OUT.read_bytes()).hexdigest()
    print(f"  저장: {OUT.relative_to(NB_ROOT)}")
    print(f"  sha256: {h[:16]}...")

    if CHAMPION.exists():
        ref = pd.read_csv(CHAMPION, encoding="utf-8-sig")
        md = max(float(np.abs(sub[g].to_numpy() - ref[g].to_numpy()).max())
                 for g in CAPACITY_KWH)
        same = hashlib.sha256(CHAMPION.read_bytes()).hexdigest() == h
        status = "파일 동일" if same else "일치" if md < 1e-6 else "불일치"
        print(f"\n  최종 제출 파일과 비교: 최대 차이 {md:.6e} kWh ({status})")

    print(f"\n완료 ({time.time() - t0:.0f}s)")

## 3. 실행

`open/probe_month_g1m10s95.csv`(최종 제출 파일)가 있으면 결과와 비교해서 최대 차이를 출력한다.

In [4]:
main()

추론 코드 (inference) — 학습 없음, 저장된 모델만 사용

[1/4] 평가 기간 피처 구축...


  X_test: (8760, 295)

[2/4] 모델 예측 + 보정층...


  kpx_group_1: 평균 41.84%cap  최대 98.53%cap


  kpx_group_2: 평균 43.29%cap  최대 98.26%cap


  kpx_group_3: 평균 38.91%cap  최대 98.64%cap

[3/4] 사후 변환 체인...


  kpx_group_1: 평균 41.08%cap  최대 96.49%cap
  kpx_group_2: 평균 44.04%cap  최대 98.56%cap
  kpx_group_3: 평균 34.96%cap  최대 86.52%cap

[4/4] 검증 및 저장...
  형식 확인: 8,760행, id 일치, 결측 없음, 0 ~ 설비용량 범위
  저장: open\final_submission.csv
  sha256: 45a54ddfa7e2d001...

  최종 제출 파일과 비교: 최대 차이 7.275958e-12 kWh (일치)

완료 (13s)
